# General 1D CSD pipeline

Works for:
1. **Stim / Lombard** folders (full or partial probe)
2. **Delta peak/trough** exports from `Estimate_delta_peak_trough_csd.m`
   - NPX1: `<animal>/peak|trough/` (full channel count)
   - NPX2: `<animal>/probeN/peak|trough/` (one shank, fewer channels)

No re-reference / no baseline for delta runs (`DELTA_CSD_DEFAULTS`).


In [ ]:
# %pip install numpy scipy matplotlib kcsd


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from csd_1d_pipeline import (
    DELTA_CSD_DEFAULTS,
    EXAMPLE_DATASETS,
    discover_delta_csd_datasets,
    merge_config,
    run_dataset,
    run_many,
)

print('Pipeline ready.')
print('Delta defaults:', {k: DELTA_CSD_DEFAULTS[k] for k in ['t_pre','t_post','ref_chan','baseline_correct','normalize']})
print('Expect MATLAB tree: <root>/peak|trough/<animal>/[probeN]/')


## A) Delta peak/trough CSD (recommended path)

Point `DELTA_ROOT` at the MATLAB export folder, then run.


In [ ]:
DELTA_ROOT = r"//experimentfs.bccn-berlin.pri/experiment/PlayNeuralData/NPX-OPTO PLAY NMM/PlayBout Analysis/play bout analysis behavior and physiology/Data/Analysis results/delta peak trough CSD"

DATASETS = discover_delta_csd_datasets(DELTA_ROOT)
print(f"Found {len(DATASETS)} folders")
for d in DATASETS[:10]:
    print(" ", d["name"], "->", d["folder"])
if len(DATASETS) > 10:
    print(" ...")


In [ ]:
# Optional later: DELTA_CSD_DEFAULTS["normalize"] = "zscore_channel"
results = run_many(DATASETS, defaults=DELTA_CSD_DEFAULTS, stop_on_error=False) if DATASETS else []
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]
print(f"Finished: {len(ok)} ok, {len(bad)} failed")
for r in bad[:20]:
    print(" -", r["name"], ":", r["error"])


## B) Manual / Lombard stim list

Use this if you are not using the delta export tree.


In [ ]:
DEFAULTS = dict(
    fs=2500.0,
    t_pre=200.0,
    t_post=800.0,
    step_ms=5.0,
    ref_chan="deepest",
    baseline_correct=True,
    normalize="none",
    compute_kcsd=True,
    compute_fd=True,
    y_scale="mm",
    h=0.25,
    n_src_factor=4.0,
    save=True,
    save_plots=True,
    plot=True,
)

# DATASETS = EXAMPLE_DATASETS
# results = run_many(DATASETS, defaults=DEFAULTS, stop_on_error=False)


## Outputs

Per input folder: `csd_1d/` plus `csd_1d_array.npy` / `csd_fd_array.npy`.

CLI:

```bash
python csd_1d_pipeline.py --delta-root "/path/to/delta peak trough CSD"
```

